In [1]:
!nvidia-smi

Thu Oct  8 07:07:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
from numba import cuda
import os
device = cuda.get_current_device()
print(f"Device Name: {device.name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y},{device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024,64
Warp Size: 32


In [13]:
import numpy as np
import time
from numba import cuda
# 1. CPU Reference Implementation (Pure Python loop)
def double_cpu_pure(arr):
  out = np.empty_like(arr)
  for i in range(len(arr)):
    out[i] = arr[i] * 2.0
  return out
# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
# Calculate unique 1D thread ID
  idx = cuda.grid(1)
  if idx < d_in.shape[0]:
    d_out[idx] = d_in[idx] * 2.0
def benchmark_run(N, student_id_seed=1234):
  np.random.seed(student_id_seed)
  h_in = np.random.rand(N).astype(np.float32)
  # Measure CPU Pure Loop (skip for N >= 10M if too slow)
  if N <= 1_000_000:
    t0 = time.perf_counter()
    _ = double_cpu_pure(h_in)
    cpu_time_ms = (time.perf_counter() - t0) * 1000.0
  else:
    cpu_time_ms = float('nan') # Extrapolate or skip

  # GPU Setup
  threads_per_block = 256
  blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

  # Warmup compilation run
  d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
  double_gpu_kernel[1, 32](d_warmup, d_warmup)
  cuda.synchronize()

  # GPU Roundtrip Measurement (Includes PCIe transfers)
  t0 = time.perf_counter()
  d_in = cuda.to_device(h_in)
  d_out = cuda.device_array_like(d_in)
  double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
  h_out = d_out.copy_to_host()
  cuda.synchronize()
  gpu_total_ms = (time.perf_counter() - t0) * 1000.0

  # GPU Compute-Only Measurement (Pre-allocated in VRAM)
  t0 = time.perf_counter()
  double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
  cuda.synchronize()
  gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

  return cpu_time_ms, gpu_total_ms, gpu_compute_ms

In [14]:
import pandas as pd

sizes = [100, 1000, 10000, 100000, 1000000, 5000000, 10000000]
results = []

for N in sizes:
    cpu_t, gpu_tot, gpu_comp = benchmark_run(N)
    winner = "GPU (Total)" if (not np.isnan(cpu_t) and gpu_tot < cpu_t) or np.isnan(cpu_t) else "CPU"
    results.append({
        "Array Size (N)": f"{N:,}",
        "CPU Time (ms)": f"{cpu_t:.4f}" if not np.isnan(cpu_t) else "N/A",
        "GPU Total (ms)": f"{gpu_tot:.4f}",
        "GPU Kernel Only (ms)": f"{gpu_comp:.4f}",
        "Winner (CPU vs GPU)": winner
    })

df_results = pd.DataFrame(results)
display(df_results)

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


,Array Size (N),CPU Time (ms),GPU Total (ms),GPU Kernel Only (ms),Winner (CPU vs GPU)
0,100,0.0505,1.3895,0.0870,CPU
1,"1,000",0.2564,0.5836,0.0579,CPU
2,"10,000",2.4718,0.5797,0.0577,GPU (Total)
3,"100,000",27.4825,0.6734,0.0630,GPU (Total)
4,"1,000,000",252.2677,4.6842,0.1114,GPU (Total)
5,"5,000,000",N/A,12.7141,0.2627,GPU (Total)
6,"10,000,000",N/A,24.0913,0.4651,GPU (Total)


In [16]:
# DEFECTIVE KERNEL: Notice the missing 'if idx < n' check!
@cuda.jit
def defective_kernel(d_arr):
  idx = cuda.grid(1)
  # FORBIDDEN: Writing without bounds check!
  d_arr[idx] = 999.0

# TEST HARNESS
N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

# Launch 4 blocks of 256 = 1024 threads
threads_per_block = 256
blocks_per_grid = 4

print("Launching defective kernel with 1024 threads on 1000-element array...")
defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()
print("Execution finished (or did it crash?). Checking host copy...")
h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [18]:
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
  t_idx = cuda.threadIdx.x
  b_idx = cuda.blockIdx.x
  g_idx = cuda.grid(1)
  if g_idx < log_block.shape[0]:
    log_block[g_idx] = b_idx
    log_thread[g_idx] = t_idx
    log_global[g_idx] = g_idx
    log_active[g_idx] = 1 if g_idx < N else 0

# Configuration
N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block # Exactly 5 blocks = 40 threads

log_b = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_t = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_g = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_a = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()

print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40): # Print the critical boundary transition (Block 4)
  status = "ACTIVE" if log_a[i] == 1 else "IDLE (GUARDED)"
  print(f"{log_g[i]:<10} | {log_b[i]:<10} | {log_t[i]:<10} | {status:<10}")

Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [20]:
# BUGGY NAIVE SUMMATION
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
  idx = cuda.grid(1)
  if idx < d_arr.shape[0]:
    # DATA RACE: Thousands of threads execute Read-Modify-Write at the exact same clock cycle!
    d_total[0] += d_arr[idx]

# CORRECT ATOMIC SUMMATION
@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
  idx = cuda.grid(1)
  if idx < d_arr.shape[0]:
    # HARDWARE ATOMIC LOCK: Hardware serializes updates to prevent collisions
    cuda.atomic.add(d_total, 0, d_arr[idx])